In [ ]:
# 1. Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive", force_remount=True)
MYDRIVE = Path("/content/drive/MyDrive")
PROJECT = MYDRIVE / "asr project"
assert MYDRIVE.exists(), "Google Drive did not mount correctly."
assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
print("Drive mounted correctly.")
print("Project folder:", PROJECT)

In [ ]:
# Imports
import os
import re
import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from pathlib import Path
from tqdm import tqdm

# datasets metadata
DATASETS = {
    "Pitt": {
        "wav_dir": "/content/drive/MyDrive/asr project/English/Pits/cookie/control cookie",
        "cha_dir": "/content/drive/MyDrive/asr project/Pitt-cha/Pitt/Control/cookie",
        "language": "English",
        "patient_speakers": ("PAR",)
    },
    "Greek_DemCare": {
        "wav_dir": "/content/drive/MyDrive/asr project/Greek/pilot/Patients",
        "cha_dir": "/content/drive/MyDrive/asr project/Dem@Care_cha/pilot/patients",
        "language": "Greek",
        # From inspection:
        "patient_speakers": ("PAR1","PAR2")
    },
    "Mandarin_Chou": {
        "wav_dir": "/content/drive/MyDrive/asr project/Mandarin/picture_description/HC",
        "cha_dir": "/content/drive/MyDrive/asr project/Chou-cha/Chou/HC",
        "language": "Mandarin",

        # From inspection:
        # PAR0 = main participant/patient speech
        # PAR1 = small extra fragments
        "patient_speakers": ("PAR0",)
    }
}

#output directory
OUTPUT_ROOT = PROJECT / "preprocessed_patient_audio/GreekPatientsNew"
TARGET_SR = 16000
MIN_SEGMENT_SECONDS = 0.05
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Output root:", OUTPUT_ROOT)

In [ ]:
def extract_patient_intervals_from_cha(
    cha_path,
    patient_speakers=("PAR",),
):
    """
    extract patient speech intervals from a chat or cha file

    looks for lines such as:
        *PAR: text ... \x1512345_18420\x15
        *PAR2: text ... \x1512345_18420\x15
        *PAR0: text ... \x1512345_18420\x15
    returns:
        intervals: list of start and end times in seconds
    """
    #convert the input path to a path object
    cha_path = Path(cha_path)
    # read the transcript
    text = cha_path.read_text(
        encoding="utf-8",
        errors="replace"
    )
    intervals = []
    # match chat time markers containing millisecond values
    time_pattern = re.compile(r"\x15(\d+)_(\d+)\x15")
    # for each transcript line
    for line in text.splitlines():
        line = line.strip()
        # skip lines without a speaker marker
        if not line.startswith("*"):
            continue
        # skip malformed speaker lines
        if ":" not in line:
            continue
        # extract the speaker identifier
        speaker = line.split(":", 1)[0].replace("*", "").strip()
        # keep only selected patient speakers
        if speaker not in patient_speakers:
            continue
        # find all time markers in the line
        matches = time_pattern.findall(line)
        # convert each interval from milliseconds to seconds
        for start_ms, end_ms in matches:
            start_sec = int(start_ms) / 1000.0
            end_sec = int(end_ms) / 1000.0
            # keep only intervals with positive duration
            if end_sec > start_sec:
                intervals.append((start_sec, end_sec))
    # sort intervals by their start time
    intervals = sorted(intervals, key=lambda x: x[0])
    return intervals

In [ ]:
# audio loading and patient-only extraction

def load_wav_mono_16k_np(wav_path, target_sr=16000):
    """
    Load WAV as mono, 16 kHz, float32, peak-normalized.
    """
    # load resample and convert the audio to mono
    x, sr = librosa.load(
        wav_path,
        sr=target_sr,
        mono=True
    )
    # convert the waveform to float32
    x = x.astype(np.float32)

    # find the maximum absolute amplitude
    max_abs = np.max(np.abs(x)) if len(x) > 0 else 0.0

    # normalize non silent audio
    if max_abs > 0:
        x = x / max_abs
    # return the waveform and target sample rate
    return x, target_sr


def make_patient_only_audio(
    wav_path,
    intervals,
    target_sr=16000,
    min_segment_seconds=0.05,
):
    """
    Create one concatenated waveform containing only patient speech
    """
    # load the complete recording
    x, sr = load_wav_mono_16k_np(
        wav_path,
        target_sr=target_sr
    )
    patient_segments = []
    used_intervals = []
    # extract each patient speech interval
    for start_sec, end_sec in intervals:
        duration = end_sec - start_sec
        # skip intervals that are too short
        if duration < min_segment_seconds:
            continue
        # convert interval times to sample positions
        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)

        # keep sample positions inside the waveform
        start_sample = max(0, start_sample)
        end_sample = min(len(x), end_sample)

        # keep intervals with valid sample ranges
        if end_sample > start_sample:
            segment = x[start_sample:end_sample]
            patient_segments.append(segment)
            used_intervals.append((start_sec, end_sec))

    # stop when no valid patient segments exist
    if len(patient_segments) == 0:
        raise ValueError("No valid patient speech segments found.")

    # join all patient segments
    patient_audio = np.concatenate(patient_segments)

    # find the maximum amplitude of the combined audio
    max_abs = np.max(np.abs(patient_audio)) if len(patient_audio) > 0 else 0.0

    # normalize the combined patient audio
    if max_abs > 0:
        patient_audio = patient_audio / max_abs

    # calculate the original recording duration
    original_duration = len(x) / sr

    return patient_audio.astype(np.float32), sr, used_intervals, original_duration

In [ ]:
# robust file matching
def build_file_map(directory, extensions):
    """
    Create dictionary:
        file stem -> file path
    Handles lowercase and uppercase extensions.
    """

    directory = Path(directory)

    if isinstance(extensions, str):
        extensions = [extensions]

    files = []
    # search recursively for each extension
    for ext in extensions:
        ext = ext.lower().lstrip(".")
        files.extend(directory.rglob(f"*.{ext}"))
        files.extend(directory.rglob(f"*.{ext.upper()}"))

    files = sorted(set(files))

    return {f.stem: f for f in files}

In [ ]:
# main preprocessing function

def preprocess_all_datasets(
    datasets,
    output_root,
    target_sr=16000,
    min_segment_seconds=0.05,
    overwrite=True,
):
    """
    Preprocess all datasets:
        - match WAV and CHA files
        - extract patient intervals
        - save patient-only WAV
        - save interval JSON
        - save metadata CSV
    """
    #prepare output folder
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    all_metadata = []

    # process each dataset
    for dataset_name, config in datasets.items():
        print(f"\nProcessing dataset: {dataset_name}")

        wav_dir = Path(config["wav_dir"])
        cha_dir = Path(config["cha_dir"])
        language = config["language"]
        patient_speakers = config.get("patient_speakers", ("PAR",))

        print("WAV dir exists:", wav_dir.exists())
        print("CHA dir exists:", cha_dir.exists())
        print("Patient speakers:", patient_speakers)

        # create dataset output folders
        dataset_out_dir = output_root / dataset_name
        audio_out_dir = dataset_out_dir / "wav_patient_only"
        interval_out_dir = dataset_out_dir / "intervals"

        audio_out_dir.mkdir(parents=True, exist_ok=True)
        interval_out_dir.mkdir(parents=True, exist_ok=True)

        # map file identifiers to wav and cha paths
        wav_map = build_file_map(wav_dir, [".wav"])
        cha_map = build_file_map(cha_dir, [".cha"])

        # keep identifiers available in both folders
        common_ids = sorted(set(wav_map.keys()) & set(cha_map.keys()))

        print("WAV files:", len(wav_map))
        print("CHA files:", len(cha_map))
        print("Matched pairs:", len(common_ids))

        # find files without matching transcript or audio
        missing_cha = sorted(set(wav_map.keys()) - set(cha_map.keys()))
        missing_wav = sorted(set(cha_map.keys()) - set(wav_map.keys()))

        # show wav files without cha files
        if len(missing_cha) > 0:
            print("WAV files without CHA:", len(missing_cha))
            print("Example missing CHA:", missing_cha[:5])

        # show cha files without wav files
        if len(missing_wav) > 0:
            print("CHA files without WAV:", len(missing_wav))
            print("Example missing WAV:", missing_wav[:5])

        # process every matched wav and cha pair
        for file_id in tqdm(common_ids):
            wav_path = wav_map[file_id]
            cha_path = cha_map[file_id]

            # define output paths
            out_wav_path = audio_out_dir / f"{file_id}_patient.wav"
            out_json_path = interval_out_dir / f"{file_id}_intervals.json"

            try:
                # extract patient speech timestamps
                intervals = extract_patient_intervals_from_cha(
                    cha_path=cha_path,
                    patient_speakers=patient_speakers
                )

                if len(intervals) == 0:
                    raise ValueError("No patient intervals found in CHA.")

                # create the patient-only waveform
                patient_audio, sr, used_intervals, original_duration = make_patient_only_audio(
                    wav_path=wav_path,
                    intervals=intervals,
                    target_sr=target_sr,
                    min_segment_seconds=min_segment_seconds
                )

                # calculate patient speech duration
                patient_duration = len(patient_audio) / sr

                # calculate the patient speech percentage
                if original_duration > 0:
                    patient_percentage = 100 * patient_duration / original_duration
                else:
                    patient_percentage = np.nan

                # save patient-only audio
                if overwrite or not out_wav_path.exists():
                    sf.write(
                        file=str(out_wav_path),
                        data=patient_audio,
                        samplerate=sr
                    )

                # prepare interval metadata
                interval_data = {
                    "dataset": dataset_name,
                    "language": language,
                    "file_id": file_id,
                    "wav_path": str(wav_path),
                    "cha_path": str(cha_path),
                    "output_wav_path": str(out_wav_path),
                    "target_sr": sr,
                    "patient_speakers": list(patient_speakers),
                    "n_intervals_raw": len(intervals),
                    "n_intervals_used": len(used_intervals),
                    "intervals": [
                        {
                            "start_sec": float(s),
                            "end_sec": float(e)
                        }
                        for s, e in used_intervals
                    ]
                }

                if overwrite or not out_json_path.exists():
                    with open(out_json_path, "w", encoding="utf-8") as f:
                        json.dump(
                            interval_data,
                            f,
                            indent=2,
                            ensure_ascii=False
                        )
                # record successful preprocessing metadata
                all_metadata.append({
                    "dataset": dataset_name,
                    "language": language,
                    "file_id": file_id,
                    "original_wav_path": str(wav_path),
                    "cha_path": str(cha_path),
                    "patient_wav_path": str(out_wav_path),
                    "interval_json_path": str(out_json_path),
                    "sample_rate": sr,
                    "patient_speakers": ",".join(patient_speakers),
                    "n_intervals_raw": len(intervals),
                    "n_intervals_used": len(used_intervals),
                    "original_duration_sec": original_duration,
                    "patient_duration_sec": patient_duration,
                    "patient_percentage": patient_percentage,
                    "status": "ok",
                    "error": ""
                })
            # record errors without stopping other files
            except Exception as e:
                all_metadata.append({
                    "dataset": dataset_name,
                    "language": language,
                    "file_id": file_id,
                    "original_wav_path": str(wav_path),
                    "cha_path": str(cha_path),
                    "patient_wav_path": "",
                    "interval_json_path": "",
                    "sample_rate": target_sr,
                    "patient_speakers": ",".join(patient_speakers),
                    "n_intervals_raw": np.nan,
                    "n_intervals_used": np.nan,
                    "original_duration_sec": np.nan,
                    "patient_duration_sec": np.nan,
                    "patient_percentage": np.nan,
                    "status": "error",
                    "error": str(e)
                })

    # combine all processing records
    metadata_df = pd.DataFrame(all_metadata)

    # save the complete preprocessing metadata
    metadata_path = output_root / "preprocessing_metadata.csv"
    metadata_df.to_csv(metadata_path, index=False)

    print("\nSaved metadata to:", metadata_path)
    print("Total files:", len(metadata_df))

    # show success and error counts
    if len(metadata_df) > 0 and "status" in metadata_df.columns:
        print("Successful:", (metadata_df["status"] == "ok").sum())
        print("Errors:", (metadata_df["status"] == "error").sum())

        print("\nBy dataset/status:")
        print(metadata_df.groupby(["dataset", "status"]).size())
    else:
        print("No files were processed.")

    return metadata_df

In [ ]:
# run preprocessing
metadata_df = preprocess_all_datasets(
    datasets=DATASETS,
    output_root=OUTPUT_ROOT,
    target_sr=TARGET_SR,
    min_segment_seconds=MIN_SEGMENT_SECONDS,
    overwrite=True
)
metadata_df.head()

In [ ]:
# check dataset statistics

ok_df = metadata_df[metadata_df["status"] == "ok"].copy()

summary = ok_df.groupby(["dataset", "language"]).agg(
    n_files=("file_id", "count"),
    mean_original_duration_sec=("original_duration_sec", "mean"),
    median_original_duration_sec=("original_duration_sec", "median"),
    mean_patient_duration_sec=("patient_duration_sec", "mean"),
    median_patient_duration_sec=("patient_duration_sec", "median"),
    mean_patient_percentage=("patient_percentage", "mean"),
    median_patient_percentage=("patient_percentage", "median"),
).reset_index()

summary